# Lecture 4: value function iteration

**ECU44291 Quantitative Macroeconomics, Week 4.**

Drop the age from last week's Bellman equation and there is no terminal condition to start from. The equation becomes a fixed-point problem in a *function*:
$$V(k) = \max_{0 < k' < k^{\alpha}} \; \ln(k^{\alpha} - k') + \beta V(k').$$
This is the neoclassical growth model with log utility and full depreciation. It has a closed form, $V(k) = A + B\ln k$ and $g(k) = \alpha\beta k^{\alpha}$, which is our check.

Value function iteration is Week 1's fixed-point loop applied to $V$: guess, apply the operator, repeat until nothing changes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

alpha, beta = 0.3, 0.9
k = np.linspace(0.02, 0.5, 200)

def closed_form_value(k):
    B = alpha / (1 - alpha * beta)
    A = (1 / (1 - beta)) * ((alpha * beta / (1 - alpha * beta)) * np.log(alpha * beta) + np.log(1 - alpha * beta))
    return A + B * np.log(k)

def closed_form_policy(k):
    return alpha * beta * k ** alpha

## Warm-up: two iterations by hand

The slide's cake: sizes $w \in \{0, 1, 2\}$, $u(c) = \sqrt{c}$, $\beta = 0.9$, $V_0 = 0$. Do iterations 1 and 2 on paper first, then run the cell to check. Every infeasible choice (keeping more than the cake) gets $-\infty$.

In [ ]:
w = np.array([0, 1, 2])
c_cake = w[:, None] - w[None, :]                  # rows: w today; columns: w' kept
u_cake = np.full(c_cake.shape, -np.inf)
u_cake[c_cake >= 0] = np.sqrt(c_cake[c_cake >= 0])
V_cake = np.zeros(3)
for it in (1, 2, 3):
    value = u_cake + 0.9 * V_cake[None, :]
    V_cake, keep = value.max(axis=1), value.argmax(axis=1)
    print(f"iteration {it}: V = {np.round(V_cake, 3)}, keep w' = {keep}")

## Task 1: the Bellman operator

Compute the utility matrix `U[i, j] = ln(k_i^alpha - k_j)` once (with $-\infty$ where consumption is not positive). Then write `bellman(V)` returning `(TV, policy)` where `TV[i] = max_j U[i, j] + beta * V[j]`. One line each with NumPy broadcasting. Why compute `U` once rather than inside the loop?

In [ ]:
# SOLUTION
c = k[:, None] ** alpha - k[None, :]
U = np.full(c.shape, -np.inf)
U[c > 0] = np.log(c[c > 0])

def bellman(V):
    value = U + beta * V[None, :]
    return value.max(axis=1), value.argmax(axis=1)

TV, pol = bellman(np.zeros(len(k)))
print(TV[:3], pol[:3])

## Task 2: value function iteration

Write the loop: start from $V_0 = 0$, apply `bellman`, measure the sup-norm distance, stop below $10^{-6}$. Keep every iterate in a list so we can plot them. Report the number of iterations. Then plot the iterates and the closed form on one figure.

In [ ]:
# SOLUTION
def vfi(tol=1e-6, max_iter=10000):
    V = np.zeros(len(k))
    history, dists = [V.copy()], []
    for it in range(max_iter):
        V_new, policy = bellman(V)
        dist = np.max(np.abs(V_new - V))
        dists.append(dist)
        V = V_new
        history.append(V.copy())
        if dist < tol:
            return V, policy, np.array(dists), history
    raise RuntimeError("no convergence")

V, policy, dists, history = vfi()
print(f"{len(dists)} iterations; max |V - closed form| = {np.max(np.abs(V - closed_form_value(k))):.4f}")
for Vn in history[::5]:
    plt.plot(k, Vn, color="tab:blue", lw=0.7, alpha=0.5)
plt.plot(k, closed_form_value(k), "--", color="tab:orange", lw=2, label="closed form")
plt.legend(); plt.xlabel("k"); plt.ylabel("V"); plt.show()

### What each iteration is doing

Iteration $n$ solves the $n$-period problem, so its policy should save the finite-horizon share of output, $x_n = \alpha\beta / (1 + \alpha\beta - x_{n-1})$ starting from $x = 0$. Compare at $k = 0.15$ (the table on the slide):

In [ ]:
i = np.argmin(np.abs(k - 0.15))
x = 0.0                                          # one period left: save nothing
for n in range(1, 7):
    _, pol_n = bellman(history[n - 1])           # the policy chosen at iteration n
    print(f"iteration {n}: VFI saving rate {k[pol_n[i]] / k[i] ** alpha:.3f}   n periods left {x:.3f}")
    x = alpha * beta / (1 + alpha * beta - x)    # one more period: the Euler slide's recursion

**Workflow note.** *If it converged in one iteration, it did not converge.* A first iteration from $V_0 = 0$ must change something. `git diff` shows the line that stopped it changing; `git checkout -- src/` takes it back.

## Task 3: the rate

Plot the distances on a log scale. What is the ratio of consecutive distances after the first few iterations? Compare it to $\beta$. The number of iterations to tolerance is about $\ln(\text{tol}/d_0)/\ln\beta$, with $d_0$ the first distance: check it at $\beta = 0.9$, then predict how many iterations $\beta = 0.99$ needs, and check that too.

In [ ]:
# SOLUTION
plt.semilogy(dists); plt.xlabel("iteration"); plt.ylabel("sup-norm change"); plt.show()
print("late ratios:", np.round(dists[-5:] / dists[-6:-1], 4), " beta =", beta)
print(f"beta = 0.9: predicted about {int(np.ceil(np.log(1e-6 / dists[0]) / np.log(0.9)))} iterations, actual {len(dists)}")
predicted = int(np.ceil(np.log(1e-6 / dists[0]) / np.log(0.99)))
beta_saved, beta = beta, 0.99
V99, _, d99, _ = vfi()
beta = beta_saved
print(f"beta = 0.99: predicted about {predicted} iterations, actual {len(d99)}")

## Task 4: the policy

Plot the grid policy $k' = k[\text{policy}]$ against the closed form $\alpha\beta k^{\alpha}$ and the 45-degree line. Where is the steady state? Why is the grid policy a staircase?

In [ ]:
# SOLUTION
kstar = (alpha * beta) ** (1 / (1 - alpha))
plt.step(k, k[policy], where="mid", label="grid policy")
plt.plot(k, closed_form_policy(k), "--", label="closed form")
plt.plot(k, k, "k:", lw=0.8)
plt.plot(kstar, kstar, "ko"); plt.legend(); plt.xlabel("k"); plt.ylabel("k'"); plt.show()
print(f"k* = {kstar:.4f}")

## Task 5: Howard's improvement (your exercise for next week)

Instead of improving $V$ a little each iteration, take a policy, compute its value *exactly* by solving the linear system $(I - \beta P_\sigma) v = r_\sigma$, then take the greedy policy for that value, and repeat until the policy stops changing. Write `policy_evaluation(policy)` and the loop. How many steps? We walk through it slowly at the start of the next lecture.

In [ ]:
# Solution posted next week. Try it yourself first.


## Task 6: time it

Time `vfi()`. Then write the naive version with three explicit Python loops and utility recomputed inside (the way the MSc notebook did it), run it with a 50-point grid, and time that. Measure before you optimise: which change bought the most?

In [ ]:
# SOLUTION
t0 = time.perf_counter(); vfi(); t_vfi = time.perf_counter() - t0
print(f"vectorised VFI {t_vfi:.3f} s on {len(k)} grid points")

def vfi_naive(kk, tol=1e-6):
    V = np.zeros(len(kk))
    for it in range(10000):
        V_new = np.empty(len(kk))
        for i in range(len(kk)):
            best = -np.inf
            for j in range(len(kk)):
                cc = kk[i] ** alpha - kk[j]
                if cc > 0:
                    best = max(best, np.log(cc) + beta * V[j])
            V_new[i] = best
        if np.max(np.abs(V_new - V)) < tol:
            return V_new, it + 1
        V = V_new

k50 = np.linspace(0.02, 0.5, 50)
t0 = time.perf_counter(); vfi_naive(k50); t_naive = time.perf_counter() - t0
print(f"naive loops on 50 points: {t_naive:.2f} s")

## Before next week

- **Exercise:** Task 5, Howard's improvement. Write `policy_evaluation` and the Howard loop, and check that you get the same policy as VFI. We walk through it at the start of the next lecture.
- Put `bellman` and `vfi` (and your Howard functions) in `src/vfi.py` with a test against the closed form. Commit.
- Keep working on M1 Part A. The solo window is open; Copilot is off.
- Read method card 6 (discrete VFI with Howard).
- Workflow note: if VFI converges in one iteration with distance exactly zero, it did not converge. `git diff` shows you the one line.